# Lakebase 101 — Post-Deploy

Run this **after** `databricks bundle deploy`.

1. Grants the app's service principal `CAN_MANAGE_RUN` on synced-table pipelines
2. Starts and deploys the app

In [0]:
CATALOG = "lakebase_101_catalog"
APP_NAME = "lakebase-101-app"

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineAccessControlRequest

w = WorkspaceClient()

# Get the app's service principal name
app = w.apps.get(APP_NAME)
sp_name = app.service_principal_name
print(f"App SP: {sp_name}")

# Find all synced-table pipelines for this catalog and grant permissions
granted = 0
for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"):
    try:
        w.pipelines.set_permissions(
            pipeline_id=p.pipeline_id,
            access_control_list=[
                PipelineAccessControlRequest(
                    service_principal_name=sp_name,
                    permission_level="CAN_MANAGE_RUN"
                )
            ]
        )
        granted += 1
        print(f"  ✅ {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Granted CAN_MANAGE_RUN on {granted} pipeline(s) to {sp_name}")

In [0]:
import time

# Start the app (brings up compute if stopped)
print(f"Starting app '{APP_NAME}'...")
try:
    w.apps.start(APP_NAME)
except Exception as e:
    if "already in state RUNNING" in str(e) or "ACTIVE" in str(e):
        print("  App compute already running.")
    else:
        raise

# Deploy from Git (latest commit on main)
print(f"Deploying app from Git...")
deployment = w.apps.deploy(
    APP_NAME,
    source_code_path="Lakebase-101/src/app",
    git_source={"branch": "main"}
)
print(f"  Deployment ID: {deployment.deployment_id}")
print(f"  Status: {deployment.status.state}")

# Wait for deployment to complete
print("\nWaiting for deployment...")
for _ in range(60):
    app_info = w.apps.get(APP_NAME)
    state = app_info.active_deployment.status.state if app_info.active_deployment else None
    if state and "SUCCEEDED" in str(state):
        print(f"\n✅ App deployed successfully!")
        print(f"   URL: {app_info.url}")
        break
    elif state and "FAILED" in str(state):
        print(f"\n❌ Deployment failed: {app_info.active_deployment.status.message}")
        break
    time.sleep(5)
    print(".", end="", flush=True)
else:
    print(f"\n⏳ Deployment still in progress — check the Apps UI.")